# Tutorial 05: Deep Dive: Exploring Your Results

**Before you start:** Run this from `Notebooks/tutorials/`. If you are new to the pipeline, skim [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb) first.

**In plain terms:** The vision model turns each image into an **embedding** (a numeric summary of appearance). This notebook shows typical **next steps** for digging deeper: grouping similar cells and finding patterns (clustering), identifying the most "typical" cells (prototypes), and linking the AI's complex findings back to real biological traits (explainability).

For a unified **HTML report** that bundles many of these sections automatically, use `generate_report()` and see the [Report Guide](../../docs/guides/report-guide.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

model, wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg")
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## 2. Load data, embed, optional properties

Same pattern as the other examples: discover files, run `process_images`, then optionally `compute_properties` if you linked **masks** in `find_files` (required for presets like `basic`). Adjust `reference_filters` so the **control** label matches your metadata (here: `condition == "Control"`).

In [ ]:
metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
image_dir = "path/to/images"
mask_dir = "path/to/masks" # optional, can be None if you don't have masks or if the directory doesn't exist

file_df = pheno.find_files(
    image_dir,
    metadata_fn=metadata_fn,
    mask_dir=mask_dir,
)
print(f"Found {len(file_df)} images")

pheno.process_images(wrapper, channel_mode="split")
pheno.compute_properties(property_preset="basic")

In [ ]:
# Plot number of images per group
pheno.plot_counts(group_by=["condition"])

## 3. Summaries by Experimental Group

This step checks if your physical measurements (like overall area or brightness) actually differ between your experimental groups (like 'Treated' vs. 'Untreated'). It requires that you have already run `compute_properties`.

In [ ]:
prop_keys = pheno.get_available_property_keys()

df_grouped = pheno.filter_properties_by_group(group_by=["condition"])
pheno.print_property_stats_by_group(
    df_grouped,
    properties=prop_keys[:3], # first 3 properties for brevity
)

### 3.1.1 Show top features different from reference

Shows which morphological features differ most between each group and a reference (e.g. control). Adjust `reference_group` to match your metadata.

In [ ]:
pheno.top_properties_different_from_reference(
    df_grouped,
    reference_group={"condition": "Control"},
    k=3,
)

## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/reference/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [ ]:
pheno.compute_clustering(n_clusters=5, clustering_method="kmeans")
pheno.plot_pca(color_by="cluster")

### 4.1.  Cluster Enrichment

Finding the defining traits of a group. Are cells in Cluster 1 significantly larger than everyone else? This helps you figure out exactly what biological features make a specific cluster stand out.

In [ ]:
enrichment_df = pheno.analyze_cluster_enrichment(cluster_col="cluster") # you can use any other metadata column here instead of "cluster"

print("\nCluster enrichment results:")
for cluster in sorted(enrichment_df["Cluster"].unique())[:5]:
    sub = enrichment_df[enrichment_df["Cluster"] == cluster].head(5)
    print(f"\nCluster {cluster} (top properties by |z-score|):")
    print(sub[["Property", "Score"]].to_string(index=False))

## 5. Representative / Prototype Images

Prototypes simply answer: *"Show me the most 'average' or 'typical' image for each group so I know what they look like."* It pulls the exact images that sit right in the center of their respective clusters.

In [ ]:
prototypes = pheno.find_prototypes(cluster_col="cluster", n_prototypes=3)
for name, idxs in prototypes.items():
    print(f"Cluster {name}: prototype index {idxs[0]}")
    pheno.plot_image_by_index(idxs[0], show_extra_info=False)

## 6. Outliers

**Outliers:** images unusually far from their group’s centroid good candidates for visual QC.

In [ ]:
outliers = pheno.detect_outliers(group_by="condition", threshold=3.0)
if outliers["outlier_indices"]:
    top_idx = outliers["outlier_indices"][0]
    print(f"Example outlier index: {top_idx}")
    pheno.plot_image_by_index(top_idx, show_extra_info=False)
else:
    print("No outliers detected with current settings.")

## 7. Two Kinds of Explainability

When trying to understand what the AI is actually seeing, there are two approaches:
- **1) Plot Explainers (`compute_component_correlation`):** Addresses *"Why is this image on the left side of my plot?"* It ties the axes of your visual map (like PCA/UMAP) directly to real features like cell size.
- **2) Global Explainers (`compute_embedding_property_correlations`):** Addresses *"Which physical traits influenced the AI the most overall?"* Checks general correlations across all the math the AI generated.

The interactive beginner tutorial also lets you test component–property correlations easily with sliders ([01_interactive_quickstart.ipynb](01_interactive_quickstart.ipynb)).

In [ ]:
subset_keys = pheno.get_available_property_keys()[:20]

corr_comp = pheno.compute_component_correlation(
    method="pca",
    n_components=3,
    top_k=10,
)
if corr_comp and "summary" in corr_comp and not corr_comp["summary"].empty:
    print(corr_comp["summary"].head(10).to_string(index=False))
else:
    print("No component correlation summary.")

In [ ]:
emb_corr = pheno.compute_embedding_property_correlations(property_keys=subset_keys)
emb_agg = pheno.aggregate_embedding_property_correlations(emb_corr, top_k=15)
pheno.plot_property_correlations(emb_agg, top_k=15)

## 8. Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [ ]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

## See Also

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_advanced_metadata_handling.ipynb](04_advanced_metadata_handling.ipynb): Handling complex CSVs and multi-channel data
- [06_extending_phenome_plugins.ipynb](06_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers